In [8]:
from sage.repl.interface_magic import InterfaceMagic

# 1. Your working interface definition
m2 = Macaulay2(command='/opt/homebrew/bin/M2')

# 2. Get the current Jupyter notebook shell
shell = get_ipython()

# 3. Bind the 'm2' object to a Jupyter magic interface
interface = InterfaceMagic("m2", m2)

# 4. Register %%m2 as a recognized cell magic
shell.register_magic_function(interface.cell_magic_factory(), magic_name="m2", magic_kind='cell')

In [9]:
m2('loadPackage "NormalToricVarieties"')
m2('loadPackage "RankThreeTorics"')
m2('loadPackage "ToricExtras"')
m2('loadPackage "Topcom"')

RankThreeTorics

In [10]:
%%m2
loadedPackages
installedPackages()

{RankThreeTorics, ToricExtras, NormalToricVarieties, Truncations, Polyhedra, Varieties, Isomorphism, Saturation, Elimination, Complexes, Schubert2, TangentCone, SimpleDoc, ReesAlgebra, PrimaryDecomposition, MinimalPrimes, PackageCitations, OnlineLookup, LLLBases, InverseSystems, IntegralClosure, PushForward, ConwayPolynomials, Classic, Core}

List

{NormalToricVarieties, Polyhedra, RankThreeTorics, ToricExtras}

List


In [11]:
%%m2

indextovector = (i, X) -> (
    R = rays X;
    return R#i
);

checkvectorincone = (u,c,X) -> (
    coneinvectors = for i in c list indextovector(i, X);
    coneobject = coneFromVData transpose matrix coneinvectors;
    uMatrix = transpose matrix {u};
    return inInterior(uMatrix,coneobject)
);

-- input u = {1,1,1}, output c = {1,2,3}
vectortocone = (u,X) -> (
    for i in (0,1,2,3) do (
        for c in (orbits X)#i do (
            if checkvectorincone(u,c,X) == true then (
                return c;
            )
        )
    )
);

primitivePoints = (X,n) -> (
    pts = {};
    R = rays X;
    for a from -n to n do
      for b from -n to n do
        for c from -n to n do (
          if gcd(a, gcd(b,c)) == 1 and not member({a,b,c}, R) then pts = append(pts, {a,b,c});
        );
    return pts
);

primitivePointspos = (X,n) -> (
    pts = {};
    R = rays X;
    for a from 1 to n do
      for b from 1 to n do
        for c from 1 to n do (
          if gcd(a, gcd(b,c)) == 1 and not member({a,b,c}, R) then pts = append(pts, {a,b,c});
        );
    return pts
);


isominlist = (X,L) -> (
    for Y in L do if areIsomorphic(X,Y) == true then return true;
    return false
);

removeisomorphic = L -> (
    newlist = {};
    for X in L do if isominlist(X,newlist) == false then newlist = append(newlist,X);
    return newlist
);

doblowupofvector = (X,v) -> (
    c = vectortocone(v,X);
    return toricBlowup(c,X,v)
);

blowuplist = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = for p in points list doblowupofvector(X,p);
    return removeisomorphic(outputlist)
);

Picardnumberblowups = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplist (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = removeisomorphic(templist);
    );
    return book
);

blowuplistwoiso = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = for p in points list doblowupofvector(X,p);
    return outputlist
);

Picardnumberblowupswoiso = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwoiso (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

blowuplistwoisosmooth = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = {};
    for p in points do (
        B = doblowupofvector(X,p);
        if isSmooth B then outputlist = append(outputlist, B);
    );
    return outputlist
);

Picardnumberblowupswoisosmooth = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwoisosmooth (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

getFVector = X -> apply(dim X + 1, i -> length (orbits X)#i);
                        
fastRemoveIsomorphic = L -> (
    bins = new MutableHashTable;
    
    -- Step 1: Group varieties by their f-vector
    for X in L do (
        inv = getFVector(X);
        if not bins#?inv then bins#inv = {};
        bins#inv = append(bins#inv, X);
    );

    uniqueList = {};
    
    -- Step 2: Check for isomorphisms ONLY within each bin
    for inv in keys bins do (
        bin = bins#inv;
        uniqueInBin = {};
        
        for X in bin do (
            isIso = false;
            for Y in uniqueInBin do (
                if areIsomorphic(X,Y) then (
                    isIso = true;
                    break; -- We found a match, stop checking this bin
                );
            );
            if not isIso then uniqueInBin = append(uniqueInBin, X);
        );
        uniqueList = join(uniqueList, uniqueInBin);
    );
    
    return uniqueList
);
                        
blowuplistwisosmooth = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = {};
    for p in points do (
        B = doblowupofvector(X,p);
        if isSmooth B then outputlist = append(outputlist, B);
    );
    return fastRemoveIsomorphic(outputlist)
);

Picardnumberblowupswisosmooth = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwisosmooth (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

blowuplistwisosmoothpos = (X,n) -> (
    points = primitivePointspos(X,n);
    outputlist = {};
    for p in points do (
        B = doblowupofvector(X,p);
        if isSmooth B then outputlist = append(outputlist, B);
    );
    return fastRemoveIsomorphic(outputlist)
);

Picardnumberblowupswisosmoothpos = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwisosmoothpos (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

checkIntegerSpansmoothsubv = (w, y) -> (
    -- Convert matrices to flat lists (if they aren't already)
    wList = flatten w;
    yList = flatten y;
    -- Find the index of the first non-zero entry in y
    i = 0;
    while yList#i == 0 do i = i + 1;
    -- In Macaulay2, '%' works perfectly on plain integers!
    -- If the remainder is 0, 'a' is an integer.
    isInteger = (wList#i % yList#i == 0);
    if isInteger then (
        -- '//' also works perfectly on plain integers for exact division
        -- a = wList#i // yList#i; 
        -- print("a is the integer: " | toString a);
        return true;
    ) else (
        -- print("a is a fraction!");
        return false;
    )
);


smoothsubv = (P,X) -> (
    if computel(P,X) == 1 then(
        pvectors = for i in P list indextovector(i,X);
        w = sum pvectors;
        c = vectortocone(w,X);
        y = for j in c list indextovector(j,X);
        return checkIntegerSpansmoothsubv(w,y);
    ) else return false;
);
                                       
                                       -- P = {3,0}
relationssum = (P,X) -> (
    pvectors = for i in P list indextovector(i,X);
    w = sum pvectors;
    if w == {0,0,0} then return pi;
    c = vectortocone(w,X);
    cvectors = for j in c list indextovector(j,X);
    combinedmatrix = join(pvectors,cvectors);
    M = transpose matrix combinedmatrix;
    rltns = ker M;
    G = gens rltns;
    total = sum flatten entries G;
    return total
);

computel = (P,X) -> (
    pvectors = for i in P list indextovector(i,X);
    w = sum pvectors;
    if w == {0,0,0} then return 0;
    c = vectortocone(w,X);
    return length c
);

numberofflops = X -> (
    fcounter = 0;
    PCS = toricPrimitiveCollections(rays X, max X);
    for PC in PCS do (if (length PC == 2 and relationssum(PC,X) == 0) then fcounter=fcounter+1);
    return fcounter
);

numberofblowdownsnotfloips = X -> (
    bcounter = 0;
    PCS = toricPrimitiveCollections(rays X, max X);
    for PC in PCS do if computel(PC,X) == 1 then bcounter=bcounter+1;
    return bcounter
);

numberofflopsandblowdownsnotfloips = X -> (
    fcounter = 0;
    bcounter = 0;
    PCS = toricPrimitiveCollections(rays X, max X);
    for PC in PCS do (if (length PC == 2 and relationssum(PC,X) == 0) then fcounter=fcounter+1);
    for PC in PCS do if (computel(PC,X) == 1 and smoothsubv(PC,X)) then bcounter=bcounter+1;
    return (fcounter, bcounter)
);
                                       
                                       coneExistenceCheck = (S, fan) -> (
for cone in fan do (
if isSubset(S, cone) then (
return true;
);
);
return false;
);

properSubsetCheck = (S, fan) -> (
for ray in S do (
if coneExistenceCheck(S-set{ray}, fan) == false then (
return false;
);
);
return true;
);

isPrimitiveCollection = (P, Var) -> (
    if coneExistenceCheck(P, (orbits Var)#0) then (
        return false;
    ) else (
        return properSubsetCheck(P, (orbits Var)#0);
    );
);
    
supsetsOfPrimColl = (E, B) -> (
return set{for P in E-set{B} when isSubset(B, P) list P};
);
    
primitiveCollectionss = (Var) -> (
n = length rays Var;
primColls = select(subsets(toList(0..n-1)), x -> length x > 1);
for P in subsets(toList(0..n-1), 2) do (
if coneExistenceCheck(P, orbits(Var, 0)) == false then (
primColls = primColls - supsetsOfPrimColl(primColls, P);)
else (
primColls = primColls - set{P};
);
);
for i in toList(3..n) do (
for P in subsets(toList(0..n-1), i) do (
if member(P, primColls) == false then continue;
if isPrimitiveCollection(P, Var) then (
primColls = primColls - supsetsOfPrimColl(primColls, P);
) else (
primColls = primColls - set{P};
);
);
);
return sort primColls;
);
    
countflopsandsmoothblowdowns = X -> (
    fcounter = 0;
    bcounter = 0;
    PCS = primitiveCollectionss(X);
    for PC in PCS do (if (length PC == 2 and relationssum(PC,X) == 0) then fcounter=fcounter+1);
    for PC in PCS do if (computel(PC,X) == 1) then bcounter=bcounter+1;
    return (fcounter, bcounter)
);

In [12]:
%%m2

Y = toricProjectiveSpace 3;
L = Picardnumberblowupswisosmooth(Y,4,2)

MutableHashTable{...4...}

MutableHashTable


In [13]:
%%m2

print length L#2
print length L#3
print length L#4
--print length L#5


2
11
124


In [14]:
%%m2

for Z in L#4 do(
    count = countflopsandsmoothblowdowns(Z);
    if count#1 < 4 then (
        print count;
        print (rays Z, max Z);
    )
)

(1, 3)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}}, {{0, 1, 2}, {0, 1, 4}, {0, 2, 5}, {0, 4, 5}, {1, 2, 3}, {1, 3, 4}, {2, 3, 5}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}})
(2, 3)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {1, 1, 0}, {-1, 0, -1}}, {{0, 1, 4}, {0, 1, 5}, {0, 4, 6}, {0, 5, 6}, {1, 3, 4}, {1, 3, 5}, {2, 3, 4}, {2, 3, 5}, {2, 4, 6}, {2, 5, 6}})
(1, 3)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, 0, 0}, {-1, -1, 0}, {-2, -1, 1}}, {{0, 1, 2}, {0, 1, 5}, {0, 2, 4}, {0, 4, 5}, {1, 2, 3}, {1, 3, 5}, {2, 3, 4}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}})
(3, 3)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, 0, 0}, {0, -1, 0}, {0, 0, -1}}, {{0, 1, 5}, {0, 1, 6}, {0, 2, 4}, {0, 2, 6}, {0, 3, 4}, {0, 3, 5}, {1, 2, 3}, {1, 2, 6}, {1, 3, 5}, {2, 3, 4}})


In [15]:
%%m2

Z41 = normalToricVariety({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}}, {{0, 1, 2}, {0, 1, 4}, {0, 2, 5}, {0, 4, 5}, {1, 2, 3}, {1, 3, 4}, {2, 3, 5}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}})
Z42 = normalToricVariety({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {1, 1, 0}, {-1, 0, -1}}, {{0, 1, 4}, {0, 1, 5}, {0, 4, 6}, {0, 5, 6}, {1, 3, 4}, {1, 3, 5}, {2, 3, 4}, {2, 3, 5}, {2, 4, 6}, {2, 5, 6}})
Z43 = normalToricVariety({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, 0, 0}, {-1, -1, 0}, {-2, -1, 1}}, {{0, 1, 2}, {0, 1, 5}, {0, 2, 4}, {0, 4, 5}, {1, 2, 3}, {1, 3, 5}, {2, 3, 4}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}})
Z44 = normalToricVariety({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, 0, 0}, {0, -1, 0}, {0, 0, -1}}, {{0, 1, 5}, {0, 1, 6}, {0, 2, 4}, {0, 2, 6}, {0, 3, 4}, {0, 3, 5}, {1, 2, 3}, {1, 2, 6}, {1, 3, 5}, {2, 3, 4}})

Z41

NormalToricVariety

Z42

NormalToricVariety

Z43

NormalToricVariety

Z44

NormalToricVariety


In [16]:
%%m2

L41 = Picardnumberblowupswisosmooth(Z41,2,2)

MutableHashTable{...2...}

MutableHashTable


In [17]:
%%m2

for Z in L41#2 do(
    count = countflopsandsmoothblowdowns(Z);
    if count#1 < 6 then (
        print count;
        print (rays Z, max Z);
    )
)

(3, 5)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}, {-2, -2, -1}}, {{0, 1, 2}, {0, 1, 7}, {0, 2, 5}, {0, 5, 7}, {1, 2, 3}, {1, 3, 4}, {1, 4, 7}, {2, 3, 5}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}, {4, 5, 7}})
(2, 5)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}, {-2, -1, -1}}, {{0, 1, 2}, {0, 1, 4}, {0, 2, 7}, {0, 4, 7}, {1, 2, 3}, {1, 3, 4}, {2, 3, 5}, {2, 5, 7}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}, {4, 5, 7}})
(2, 5)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}, {-2, -1, 2}}, {{0, 1, 2}, {0, 1, 4}, {0, 2, 5}, {0, 4, 5}, {1, 2, 3}, {1, 3, 4}, {2, 3, 5}, {3, 4, 7}, {3, 5, 7}, {4, 5, 6}, {4, 6, 7}, {5, 6, 7}})
(3, 5)
({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}, {-2, 0, -1}}, {{0, 1, 2}, {0, 1, 4}, {0, 2, 7}, {0, 4, 5}, {0, 5, 7}, {1, 2, 3}, {1, 3, 4}, {2, 3, 5}, {2, 5, 7}, {3, 4, 6}, {3, 5, 6}, {4, 5, 6}})
(2, 5)
({{-1, -1, -1}, {1, 0, 

In [18]:
%%m2

Z51 = normalToricVariety({{-1, -1, -1}, {1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, 0}, {-1, 0, 0}, {-2, -1, 1}, {0, -1, 0}}, {{0, 1, 2}, {0, 1, 7}, {0, 2, 5}, {0, 4, 5}, {0, 4, 7}, {1, 2, 3}, {1, 3, 7}, {2, 3, 5}, {3, 4, 6}, {3, 4, 7}, {3, 5, 6}, {4, 5, 6}})
countflopsandsmoothblowdowns(Z51)
primitiveCollectionss(Z51)

Z51

NormalToricVariety

(1, 5)

Sequence

{{0, 3}, {0, 6}, {1, 4}, {1, 5}, {1, 6}, {2, 4}, {2, 6}, {2, 7}, {3, 4, 5}, {5, 7}, {6, 7}}

List


In [19]:
%%m2

A = primitiveBlowdown(Z51, {3, 4, 5})
countflopsandsmoothblowdowns(A)
primitiveCollectionss(A)

A

NormalToricVariety

(0, 4)

Sequence

{{0, 3}, {1, 4}, {1, 5}, {2, 4}, {2, 6}, {5, 6}}

List


In [20]:
%%m2
B = primitiveBlowdown(A, {1,4})
countflopsandsmoothblowdowns(B)
primitiveCollectionss(B)

B

NormalToricVariety

(0, 2)

Sequence

{{0, 3}, {1, 5}, {2, 4}}

List


In [21]:
%%m2
C = primitiveBlowdown(B, {2,4})
countflopsandsmoothblowdowns(C)
primitiveCollectionss(C)

C

NormalToricVariety

(0, 1)

Sequence

{{0, 3}, {1, 2, 4}}

List


In [22]:
%%m2

D = primitiveBlowdown(C, {0,3})
countflopsandsmoothblowdowns(D)
primitiveCollectionss(D)

D

NormalToricVariety

(0, 0)

Sequence

{{0, 1, 2, 3}}

List
